In [1]:
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights
from torchsummary import summary
import numpy as np
from cvat_sdk import make_client
from cvat_sdk.pytorch import ProjectVisionDataset

In [2]:
DETECTION_CLASSES = 6

class ResNet50(nn.Module):
    def __init__(self):
        super(ResNet50, self).__init__()
        

        self.conv = resnet50(weights=ResNet50_Weights.DEFAULT)
        for param in self.conv.parameters():
            param.requires_grad = False

        self.fc = nn.Sequential(
            nn.Linear(in_features = 1000, out_features = 1000, bias=True),
            nn.BatchNorm1d(1000, momentum = 0.5),
            nn.LeakyReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features = 1000, out_features = 500, bias=True),
            nn.BatchNorm1d(500, momentum = 0.5),
            nn.LeakyReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features = 500, out_features = 500, bias=True),
            nn.BatchNorm1d(500, momentum = 0.5),
            nn.LeakyReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features = 500, out_features = DETECTION_CLASSES, bias=True),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [3]:
class ClsTrainer(ResNet50):
    def __init__(self,
                lr,
                gamma,
                criterion):
        super(ClsTrainer, self).__init__()

        self.optim = torch.optim.Adam(super().parameters(), lr=lr)
        self.scheduler = torch.optim.lr_scheduler.ExponentialLR(self.optim, gamma=gamma)
        self.criterion = criterion
        self.epo_train_losses = []
        self.epo_val_losses = []

        self.train_losses = []
        self.val_losses = []

    def train_step(self, batch, target):
        self.train()
        self.optim.zero_grad()

        output = self.forward(batch)
        loss = self.criterion(output, target)

        loss.backward()
        self.optim.step()

        self.epo_train_losses.append(loss.item())

    def eval_step(self, batch, target):
        output = self.query(batch)
        loss = self.criterion(output, target)
        self.epo_val_losses.append(loss.item())

    def query(self, batch):
        self.eval()
        with torch.no_grad():
            output = self.forward(batch)

        return output

    def epoch_end(self):
        self.scheduler.step()

        self.train_losses.append(np.mean(self.epo_train_losses))
        self.val_losses.append(np.mean(self.epo_val_losses))

        self.epo_train_losses = []
        self.epo_val_losses = []

In [4]:
OBJ_SHAPE = (128,128)
class LabelTransform(torch.nn.Module):
    '''
    transforms CVAT data to proper tensor of labels
    '''
    def forward(self, Target):
        labels = []

        for data in Target.annotations.shapes:
            # лейблы в датасете начинаются с 9
            labels.append(data['label_id']-9)

        labels = torch.tensor(labels)
        labels = torch.nn.functional.one_hot(labels, DETECTION_CLASSES)

        return labels


class ImageTransform(torch.nn.Module):
    '''
    transforms CVAT data to proper tensor of bboxes features
    '''
    def forward(self, Target):
        bboxes = []

        for data in Target.annotations.shapes:

            bboxes_raw = data['points'][-4:]
            # y1, x1, y2, x2
            bboxes.append([bboxes_raw[1], bboxes_raw[0], bboxes_raw[3], bboxes_raw[2]])



        return labels, bboxes, masks


    def rle_to_mask(self, rle, bbox):
        height = int(bbox[2]-bbox[0])+1
        width  = int(bbox[3]-bbox[1])+1
        top = int(bbox[0])
        left = int(bbox[1])

        mask = np.zeros(width * height, dtype=np.uint8)

        rle.insert(0,0.0)
        rle_pairs = np.array(rle, dtype=np.uint32)

        pos = 0
        for i in range(0, len(rle_pairs), 2):
            start = pos
            end = pos + int(rle_pairs[i])
            mask[start:end] = 1  # Устанавливаем пиксели в белый цвет (255)
            pos = end + int(rle_pairs[i + 1])  # Пропускаем следующие пиксели


        return torch.unsqueeze(torch.from_numpy(mask.reshape((height, width))), dim=0)

In [5]:
with make_client(host="http://10.162.1.50:8080", credentials=('admin', 'qaedwsrf123')) as client:
    # get the dataset comprising all tasks for the Validation subset of project 12345
    dataset = ProjectVisionDataset(client, project_id=2,
                                  # transform = ImageTransform(),
                                  target_transform = LabelTransform())

MaxRetryError: HTTPConnectionPool(host='10.162.1.50', port=8080): Max retries exceeded with url: /api/server/about (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x13b4ebf40>, 'Connection to 10.162.1.50 timed out. (connect timeout=None)'))

In [3]:
import ssl
ssl._create_default_https_context = ssl._create_stdlib_context


class_weights = torch.tensor([1.5, 1.0, 1.0, 1.0, 1.0, 1.0]).to(device)
classifier_criterion = torch.nn.BCELoss(weight = class_weights)

classifier = ClsTrainer(lr = 0.0005, gamma = 0.95, criterion = classifier_criterion).to(device)

summary(classifier, (3,640,640))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 320, 320]           9,408
       BatchNorm2d-2         [-1, 64, 320, 320]             128
              ReLU-3         [-1, 64, 320, 320]               0
         MaxPool2d-4         [-1, 64, 160, 160]               0
            Conv2d-5         [-1, 64, 160, 160]           4,096
       BatchNorm2d-6         [-1, 64, 160, 160]             128
              ReLU-7         [-1, 64, 160, 160]               0
            Conv2d-8         [-1, 64, 160, 160]          36,864
       BatchNorm2d-9         [-1, 64, 160, 160]             128
             ReLU-10         [-1, 64, 160, 160]               0
           Conv2d-11        [-1, 256, 160, 160]          16,384
      BatchNorm2d-12        [-1, 256, 160, 160]             512
           Conv2d-13        [-1, 256, 160, 160]          16,384
      BatchNorm2d-14        [-1, 256, 1

In [ ]:
# ~train loop

for epo in tqdm(range(epoches)):
    for i in torch.randperm(len(train_dataset)):
        batch, targets = train_dataset[i]
        classifier.train_step(batch, targets)

    for image, targets in val_dataset:
        classifier.eval_step(batch, targets)

    classifier.epoch_end()